# Stage 8b - Compare Mole Creek and statewide intrinsic vulnerability

This comparison asks how the continuous intrinsic-vulnerability scores differ between the 25 m Mole Creek result from Stage 4.2 and the 100 m statewide result from Stage 7.6b.

The comparison uses the continuous score (0-1), not the classified maps. The Mole Creek core grid is the comparison grid, and the statewide raster is resampled onto it using nearest-neighbour, following Stage 8. Statistics and maps use cells with valid scores in both rasters. The comparison therefore covers their overlapping scored footprint within the Mole Creek core.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.mask import mask
from rasterio.io import MemoryFile
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "src"))
from karst import in_mole_creek_focus
from mapping import add_attribution, add_basemap, finish_map, raster_extent

data = Path(os.environ.get("AT3_DATA_DIR", "data"))
outpath = data / "Output_data"
project_root = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
figures_path = project_root / "figures"


## 8b.1 Resample statewide intrinsic vulnerability to the Mole Creek grid

Use nearest-neighbour resampling, as in Stage 8. This preserves the 100 m score values while placing them on the 25 m Stage 4.2 grid; it does not add 25 m detail to the statewide result.


In [ ]:
LOCAL_PATH = outpath / "intrinsic_vulnerability_v2_raw_mc_EPSG7855.tif"
STATEWIDE_PATH = outpath / "intrinsic_vulnerability_tas_100m_EPSG7855.tif"
GRID_NODATA = -9999.0

with rasterio.open(LOCAL_PATH) as src:
    if src.crs is None:
        raise ValueError(f"Mole Creek intrinsic-vulnerability raster has no CRS: {LOCAL_PATH}")
    local_masked = src.read(1, masked=True)
    local = np.asarray(local_masked.astype("float32").filled(GRID_NODATA), dtype="float32")
    local[~np.isfinite(local)] = GRID_NODATA
    local_transform = src.transform
    local_crs = src.crs
    local_shape = src.shape
    local_profile = src.profile.copy()
    local_nodata = src.nodata

with rasterio.open(STATEWIDE_PATH) as src:
    if src.crs is None:
        raise ValueError(f"Statewide intrinsic-vulnerability raster has no CRS: {STATEWIDE_PATH}")
    statewide_on_local = np.full(local_shape, GRID_NODATA, dtype="float32")
    reproject(
        source=rasterio.band(src, 1), destination=statewide_on_local,
        src_transform=src.transform, src_crs=src.crs,
        src_nodata=src.nodata, dst_transform=local_transform, dst_crs=local_crs,
        dst_nodata=GRID_NODATA, resampling=Resampling.nearest,
        init_dest_nodata=True,
    )
    statewide_on_local[~np.isfinite(statewide_on_local)] = GRID_NODATA

valid = (local != GRID_NODATA) & (statewide_on_local != GRID_NODATA)
print(f"Mole Creek grid: {local_shape}, {local_crs}, resolution {local_transform.a:g} m")
print(f"Valid overlapping comparison cells: {int(valid.sum()):,} / {local.size:,}")
if not valid.any():
    raise ValueError("The two intrinsic-vulnerability rasters have no overlapping valid cells")


## 8b.2 Agreement statistics

Report the Pearson correlation, exact score agreement, mean absolute difference, root-mean-square difference, signed mean difference, and the share of cells where each surface is higher. A positive signed difference means the local 25 m score is higher.


In [ ]:
local_values = local[valid]
statewide_values = statewide_on_local[valid]
difference = local_values - statewide_values
diff_map = np.full(local_shape, np.nan, dtype="float32")
diff_map[valid] = difference

correlation = np.corrcoef(local_values, statewide_values)[0, 1] if len(local_values) > 1 else np.nan
print(f"Valid cells: {len(local_values):,}")
print(f"Pearson r: {correlation:.3f}")
print(f"Exact score agreement: {(difference == 0).mean() * 100:.1f}%")
print(f"Mean absolute difference: {np.abs(difference).mean():.4f}")
print(f"Root-mean-square difference: {np.sqrt(np.mean(difference ** 2)):.4f}")
print(f"Mean signed difference (local - statewide): {difference.mean():.4f}")
print(f"Local higher: {(difference > 0).mean() * 100:.1f}% of cells")
print(f"Statewide higher: {(difference < 0).mean() * 100:.1f}% of cells")


## 8b.3 Visual comparison

Show both continuous surfaces on the same 0-1 colour scale and a signed difference map (local - statewide). Cells with no difference are transparent in the difference panel.


In [ ]:
bounds = tuple(gpd.read_file(outpath / "mole_creek_bounding_box_EPSG7855.gpkg").total_bounds)
fig, axes = plt.subplots(3, 1, figsize=(10, 15))
for ax, surface, title in [
    (axes[0], np.where(valid, local, np.nan), "Mole Creek intrinsic vulnerability (25 m, Stage 4.2)"),
    (axes[1], np.where(valid, statewide_on_local, np.nan), "Statewide intrinsic vulnerability resampled to 25 m"),
]:
    add_basemap(ax, bounds, margin=0, zoom=11)
    image = ax.imshow(surface, cmap="viridis", vmin=0, vmax=1, interpolation="nearest",
                      extent=raster_extent(local_transform, local_shape), zorder=2)
    fig.colorbar(image, ax=ax, shrink=0.8, pad=0.02, label="Continuous intrinsic-vulnerability score (0-1)")
    ax.set_title(title, fontsize=11)
    finish_map(ax)

ax = axes[2]
add_basemap(ax, bounds, margin=0, zoom=11)
max_abs = float(np.nanpercentile(np.abs(diff_map), 98))
if not np.isfinite(max_abs) or max_abs == 0:
    max_abs = 1e-6
shown = np.ma.masked_where(~np.isfinite(diff_map) | np.isclose(diff_map, 0), diff_map)
im = ax.imshow(shown, cmap="RdBu_r", vmin=-max_abs, vmax=max_abs, alpha=0.85,
               interpolation="nearest", extent=raster_extent(local_transform, local_shape), zorder=2)
fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02, label="Local - statewide score")
ax.set_title("Difference (local - statewide); cells with no difference are transparent", fontsize=11)
finish_map(ax)
add_attribution(axes[-1])
plt.tight_layout(rect=(0, 0.02, 1, 1))
figures_path.mkdir(parents=True, exist_ok=True)
map_path = figures_path / "intrinsic_vulnerability_scale_comparison.png"
fig.savefig(map_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved comparison map: {map_path}")


## 8b.4 Per-named-area comparison

Use pooled aggregation: polygons sharing a name are masked together. Both score rasters are compared on the same 25 m grid, and the table reports each surface's mean and valid-cell count for every named area in the Mole Creek focus.


In [ ]:
karst_mc = gpd.read_file(outpath / "karst_mc_connectivity_EPSG7855.gpkg")
karst_mc = karst_mc[karst_mc.geom_type.isin(["Polygon", "MultiPolygon"])]
karst_mc = karst_mc[(karst_mc["karst_intensity"] > 0) & in_mole_creek_focus(karst_mc)].copy()
if karst_mc.crs != local_crs:
    karst_mc = karst_mc.to_crs(local_crs)

comparison_profile = local_profile.copy()
comparison_profile.update(driver="GTiff", height=local_shape[0], width=local_shape[1], count=1,
                          dtype="float32", crs=local_crs, transform=local_transform, nodata=GRID_NODATA)
local_for_mask = np.where(valid, local, GRID_NODATA).astype("float32")
statewide_for_mask = np.where(valid, statewide_on_local, GRID_NODATA).astype("float32")
rows = []
with MemoryFile() as local_mem, MemoryFile() as statewide_mem:
    with local_mem.open(**comparison_profile) as local_ds, statewide_mem.open(**comparison_profile) as statewide_ds:
        local_ds.write(local_for_mask, 1)
        statewide_ds.write(statewide_for_mask, 1)
        for name, group in karst_mc.groupby("KNAME"):
            geoms = list(group.geometry)
            local_img, _ = mask(local_ds, geoms, crop=True, nodata=GRID_NODATA)
            statewide_img, _ = mask(statewide_ds, geoms, crop=True, nodata=GRID_NODATA)
            local_vals = local_img[0]
            local_vals = local_vals[np.isfinite(local_vals) & (local_vals != GRID_NODATA)]
            statewide_vals = statewide_img[0]
            statewide_vals = statewide_vals[np.isfinite(statewide_vals) & (statewide_vals != GRID_NODATA)]
            rows.append({
                "KNAME": name,
                "local_mean_intrinsic_vulnerability": local_vals.mean() if len(local_vals) else np.nan,
                "statewide_mean_intrinsic_vulnerability": statewide_vals.mean() if len(statewide_vals) else np.nan,
                "local_n_cells_25m": len(local_vals),
                "statewide_n_cells_25m": len(statewide_vals),
            })

comparison = pd.DataFrame(rows)
comparison["difference_local_minus_statewide"] = (
    comparison["local_mean_intrinsic_vulnerability"] - comparison["statewide_mean_intrinsic_vulnerability"]
)
comparison = comparison.sort_values("local_mean_intrinsic_vulnerability", ascending=False, na_position="last")
comparison_path = outpath / "stage8b_intrinsic_vulnerability_comparison_table.csv"
comparison.to_csv(comparison_path, index=False)
n_missing = comparison[["local_mean_intrinsic_vulnerability", "statewide_mean_intrinsic_vulnerability"]].isna().any(axis=1).sum()
print(f"Named areas with no valid comparison cells: {n_missing}")
print(f"Saved per-named-area table: {comparison_path}")
print(comparison.to_string(index=False))


## Where the scores agree and diverge

Use the overall correlation and difference statistics above to assess whether the two surfaces have a similar spatial pattern and similar score levels. A positive mean difference means the Mole Creek 25 m surface is higher on average; the per-named-area table shows where those differences occur and reports valid-cell counts alongside each mean.

A major difference between the runs is the D8 routing domain. Stage 4.2 uses the 25 m Mole Creek DEM, which covers the local study area and its 3 km buffer. Stage 7.6b uses the 100 m DEM for all of Tasmania. The local routing can therefore omit upstream terrain beyond the buffer, while the statewide routing includes it. This can change the upstream-cells-per-target-cell ratio and move a polygon across the fixed D8 class thresholds, changing its intrinsic-vulnerability score. This is a comparability limitation: differences cannot be attributed to resolution alone.

Resolution and rasterization can also contribute. The statewide values retain 100 m source resolution after nearest-neighbour resampling to the 25 m grid, so neighbouring 25 m cells can repeat the same statewide value and do not represent independent 25 m statewide estimates. The two runs also rasterize polygon inputs on different grids. Their score differences may reflect these processing and input-domain differences as well as resolution.

The differences are higher for the intrinsic vulnerability comaprison between scales than for the comparison of risk in Notebook 08. This may be becuase risk includes land-use pressure and a slope factor which can dampen connectivity differences, so the results may not be inconsistent. 

## Stage 8b outputs

- `stage8b_intrinsic_vulnerability_comparison_table.csv` - per-named-area means and valid 25 m cell counts for both continuous surfaces
- `figures/intrinsic_vulnerability_scale_comparison.png` - side-by-side continuous maps and the local-minus-statewide difference map

The comparison uses the continuous 0-1 intrinsic-vulnerability scores. The statewide values retain their 100 m source resolution after nearest-neighbour resampling; the 25 m display grid does not represent additional statewide detail.
